# 🚀 Production Vector Search

**Deploy embeddings and vector search at scale**

---

## 📋 Overview

**What you'll learn:**
- Production vector database setup
- Indexing strategies (HNSW, IVF)
- Performance optimization
- Scaling and sharding
- Monitoring and maintenance

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List, Dict
import time

print("✅ Setup complete")

## 🤔 Production Challenges

### Scale Requirements:

```
Development:
  • 1,000 vectors
  • In-memory
  • Single machine
  • Latency: who cares?

Production:
  • 10M+ vectors
  • Persistent storage
  • Distributed
  • Latency: < 100ms p99
```

### Key Metrics:

| Metric | Target | Why |
|--------|--------|-----|
| **Query latency (p50)** | < 50ms | User experience |
| **Query latency (p99)** | < 200ms | Tail latency |
| **Indexing throughput** | 1000+ docs/sec | Fast updates |
| **Recall@10** | > 95% | Quality |
| **Availability** | 99.9% | Reliability |

### Production Stack:

**Vector Databases:**
- **Pinecone**: Managed, easy, expensive
- **Weaviate**: Open-source, flexible
- **Qdrant**: Rust, fast, modern
- **Milvus**: Distributed, scalable
- **ChromaDB**: Simple, good for prototyping

## 📊 Indexing Strategies

In [ ]:
import pandas as pd

# Compare indexing strategies
indexing_comparison = pd.DataFrame([
    {
        'Method': 'Flat (Brute Force)',
        'Speed': '🐌 Slow',
        'Accuracy': '100%',
        'Memory': 'Low',
        'Best For': '< 10K vectors',
    },
    {
        'Method': 'HNSW',
        'Speed': '⚡ Fast',
        'Accuracy': '95-99%',
        'Memory': 'High',
        'Best For': '10K - 10M',
    },
    {
        'Method': 'IVF (Inverted File)',
        'Speed': '⚡⚡ Very Fast',
        'Accuracy': '85-95%',
        'Memory': 'Medium',
        'Best For': '1M - 100M+',
    },
    {
        'Method': 'PQ (Product Quantization)',
        'Speed': '⚡⚡⚡ Fastest',
        'Accuracy': '75-90%',
        'Memory': 'Very Low',
        'Best For': '100M+',
    },
])

print("📊 Vector Index Comparison\n")
print(indexing_comparison.to_string(index=False))

print("\n💡 Recommendations:")
print("  • Small (< 10K): Flat index")
print("  • Medium (10K - 1M): HNSW")
print("  • Large (1M - 100M): IVF + PQ")
print("  • Huge (100M+): Distributed HNSW or IVF")

## ⚙️ HNSW Configuration

In [ ]:
# HNSW parameters explained
hnsw_params = {
    "M": {
        "name": "Number of connections per layer",
        "default": 16,
        "range": "4-64",
        "impact": "Higher = better recall, more memory",
        "recommendation": "16-32 for most cases",
    },
    "ef_construction": {
        "name": "Size of dynamic candidate list during construction",
        "default": 200,
        "range": "100-500",
        "impact": "Higher = better index quality, slower indexing",
        "recommendation": "100-200 for speed, 200-500 for quality",
    },
    "ef_search": {
        "name": "Size of dynamic candidate list during search",
        "default": 50,
        "range": "10-500",
        "impact": "Higher = better recall, slower search",
        "recommendation": "50 for speed, 200+ for quality",
    },
}

print("⚙️  HNSW Parameter Guide\n")
print("="*70)

for param, details in hnsw_params.items():
    print(f"\n{param}:")
    print(f"  Description: {details['name']}")
    print(f"  Default: {details['default']}")
    print(f"  Range: {details['range']}")
    print(f"  Impact: {details['impact']}")
    print(f"  💡 {details['recommendation']}")

print("\n\n📊 Configuration Examples:")
print("\nFast (low latency):")
print("  M=16, ef_construction=100, ef_search=50")
print("\nBalanced:")
print("  M=32, ef_construction=200, ef_search=100")
print("\nHigh quality (best recall):")
print("  M=64, ef_construction=500, ef_search=200")

## 🏗️ Production ChromaDB Setup

In [ ]:
# Production ChromaDB configuration
print("🏗️  Production ChromaDB Setup\n")

print("""
# 1. Persistent storage
client = chromadb.PersistentClient(
    path="./vector_db",
    settings=chromadb.Settings(
        anonymized_telemetry=False,
        allow_reset=False  # Safety!
    )
)

# 2. Create collection with HNSW
collection = client.get_or_create_collection(
    name="production_vectors",
    metadata={
        "hnsw:space": "cosine",  # or 'l2', 'ip'
        "hnsw:construction_ef": 200,
        "hnsw:M": 32,
        "hnsw:search_ef": 100,
    }
)

# 3. Batch indexing (faster)
def batch_add(documents, batch_size=100):
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i+batch_size]
        collection.add(
            documents=batch,
            ids=[f"doc_{i+j}" for j in range(len(batch))]
        )
    print(f"✅ Added {len(documents)} documents")

# 4. Query with metadata filtering
results = collection.query(
    query_texts=["your query"],
    n_results=10,
    where={"category": "tech"},  # Filter by metadata
    include=["documents", "distances", "metadatas"]
)
""")

## 📈 Performance Optimization

In [ ]:
class PerformanceOptimizer:
    """Optimize vector search performance."""
    
    @staticmethod
    def benchmark_query_latency(collection, queries: List[str], k: int = 10) -> Dict:
        """Benchmark query performance."""
        
        latencies = []
        
        for query in queries:
            start = time.time()
            results = collection.query(
                query_texts=[query],
                n_results=k
            )
            latency = (time.time() - start) * 1000  # ms
            latencies.append(latency)
        
        return {
            'p50': np.percentile(latencies, 50),
            'p95': np.percentile(latencies, 95),
            'p99': np.percentile(latencies, 99),
            'mean': np.mean(latencies),
            'min': np.min(latencies),
            'max': np.max(latencies),
        }
    
    @staticmethod
    def optimize_recommendations(latency_stats: Dict) -> List[str]:
        """Recommend optimizations based on latency."""
        
        recommendations = []
        
        if latency_stats['p99'] > 200:
            recommendations.append("⚠️  P99 latency too high (> 200ms)")
            recommendations.append("  → Reduce ef_search")
            recommendations.append("  → Add caching layer")
            recommendations.append("  → Consider sharding")
        
        if latency_stats['p50'] > 50:
            recommendations.append("⚠️  Median latency high (> 50ms)")
            recommendations.append("  → Optimize HNSW parameters")
            recommendations.append("  → Check disk I/O")
            recommendations.append("  → Use faster hardware")
        
        if not recommendations:
            recommendations.append("✅ Performance looks good!")
        
        return recommendations

# Example output
print("📈 Performance Optimization Example\n")
print("""
# Benchmark
stats = optimizer.benchmark_query_latency(collection, test_queries)

Results:
  P50:  45ms  ✅
  P95:  120ms ✅
  P99:  250ms ⚠️
  Mean: 60ms

Recommendations:
  ⚠️  P99 latency too high (> 200ms)
  → Reduce ef_search from 200 to 100
  → Add Redis caching for popular queries
  → Consider query result pooling
""")

## 💾 Caching Strategy

In [ ]:
import hashlib
from typing import Optional

class QueryCache:
    """Simple in-memory cache for vector search results."""
    
    def __init__(self, max_size: int = 1000, ttl_seconds: int = 3600):
        self.cache = {}
        self.max_size = max_size
        self.ttl = ttl_seconds
    
    def _get_cache_key(self, query: str, k: int) -> str:
        """Generate cache key from query."""
        return hashlib.md5(f"{query}:{k}".encode()).hexdigest()
    
    def get(self, query: str, k: int) -> Optional[List[Dict]]:
        """Get cached results."""
        key = self._get_cache_key(query, k)
        
        if key in self.cache:
            cached_data = self.cache[key]
            
            # Check TTL
            if time.time() - cached_data['timestamp'] < self.ttl:
                return cached_data['results']
            else:
                # Expired
                del self.cache[key]
        
        return None
    
    def put(self, query: str, k: int, results: List[Dict]):
        """Cache results."""
        key = self._get_cache_key(query, k)
        
        # Evict oldest if full
        if len(self.cache) >= self.max_size:
            oldest_key = min(self.cache.items(), key=lambda x: x[1]['timestamp'])[0]
            del self.cache[oldest_key]
        
        self.cache[key] = {
            'results': results,
            'timestamp': time.time()
        }
    
    def stats(self) -> Dict:
        """Get cache statistics."""
        return {
            'size': len(self.cache),
            'max_size': self.max_size,
            'utilization': len(self.cache) / self.max_size * 100
        }

# Usage example
print("💾 Caching Example\n")
print("""
cache = QueryCache(max_size=1000, ttl_seconds=3600)

def cached_search(query: str, k: int = 10):
    # Check cache first
    cached = cache.get(query, k)
    if cached:
        print("✅ Cache hit!")
        return cached
    
    # Cache miss - query database
    print("🔍 Cache miss - querying DB")
    results = collection.query(query_texts=[query], n_results=k)
    
    # Store in cache
    cache.put(query, k, results)
    
    return results

# First query: cache miss (slow)
results1 = cached_search("machine learning")  # 45ms

# Same query: cache hit (fast!)
results2 = cached_search("machine learning")  # 0.5ms

💡 Benefits:
  - 90x faster for cached queries
  - Reduces DB load
  - Better P99 latency
""")

## 📊 Monitoring & Alerting

In [ ]:
class VectorDBMonitor:
    """Monitor vector database health and performance."""
    
    def __init__(self):
        self.metrics = {
            'query_count': 0,
            'cache_hits': 0,
            'cache_misses': 0,
            'errors': 0,
            'latencies': [],
        }
    
    def record_query(self, latency_ms: float, cache_hit: bool, error: bool = False):
        """Record query metrics."""
        self.metrics['query_count'] += 1
        self.metrics['latencies'].append(latency_ms)
        
        if cache_hit:
            self.metrics['cache_hits'] += 1
        else:
            self.metrics['cache_misses'] += 1
        
        if error:
            self.metrics['errors'] += 1
    
    def get_dashboard(self) -> Dict:
        """Get metrics dashboard."""
        latencies = self.metrics['latencies']
        total_queries = self.metrics['query_count']
        
        return {
            'total_queries': total_queries,
            'error_rate': self.metrics['errors'] / max(total_queries, 1) * 100,
            'cache_hit_rate': self.metrics['cache_hits'] / max(total_queries, 1) * 100,
            'latency_p50': np.percentile(latencies, 50) if latencies else 0,
            'latency_p95': np.percentile(latencies, 95) if latencies else 0,
            'latency_p99': np.percentile(latencies, 99) if latencies else 0,
        }
    
    def check_alerts(self) -> List[str]:
        """Check for alert conditions."""
        alerts = []
        dashboard = self.get_dashboard()
        
        if dashboard['error_rate'] > 1.0:
            alerts.append(f"🚨 High error rate: {dashboard['error_rate']:.1f}%")
        
        if dashboard['latency_p99'] > 200:
            alerts.append(f"⚠️  High P99 latency: {dashboard['latency_p99']:.0f}ms")
        
        if dashboard['cache_hit_rate'] < 50:
            alerts.append(f"⚠️  Low cache hit rate: {dashboard['cache_hit_rate']:.1f}%")
        
        return alerts

# Example
print("📊 Monitoring Dashboard Example\n")
print("""
monitor = VectorDBMonitor()

# Record queries
monitor.record_query(latency_ms=45, cache_hit=False)
monitor.record_query(latency_ms=2, cache_hit=True)
...

# Check dashboard
dashboard = monitor.get_dashboard()
print(dashboard)

Output:
{
  'total_queries': 1000,
  'error_rate': 0.1%,
  'cache_hit_rate': 75%,
  'latency_p50': 35ms,
  'latency_p95': 120ms,
  'latency_p99': 180ms,
}

# Check alerts
alerts = monitor.check_alerts()
# []
""")

## ✅ Summary

### Production Checklist:

**1. Database Setup**
```python
✅ Persistent storage (not in-memory)
✅ HNSW index configured
✅ Metadata filtering enabled
✅ Backup strategy in place
```

**2. Performance**
```python
✅ P99 latency < 200ms
✅ P50 latency < 50ms
✅ Batch indexing (not one-by-one)
✅ Caching layer
```

**3. Scaling**
```python
✅ Horizontal scaling ready
✅ Sharding strategy
✅ Load balancing
✅ Auto-scaling configured
```

**4. Monitoring**
```python
✅ Latency metrics
✅ Error rate tracking
✅ Cache hit rate
✅ Alerting configured
```

### HNSW Configuration Guide:

**Small (< 100K vectors):**
```python
M=16
ef_construction=100
ef_search=50
```

**Medium (100K - 1M):**
```python
M=32
ef_construction=200
ef_search=100
```

**Large (1M - 10M):**
```python
M=48
ef_construction=300
ef_search=150
```

### Optimization Strategies:

**1. Caching**
- Cache popular queries (20% of queries = 80% of traffic)
- Use Redis or in-memory cache
- TTL: 1-24 hours depending on data freshness

**2. Batch Operations**
```python
# DON'T: Add one at a time
for doc in documents:
    collection.add(documents=[doc])

# DO: Batch add
collection.add(documents=documents)  # All at once
```

**3. Metadata Filtering**
```python
# Pre-filter with metadata to reduce search space
results = collection.query(
    query_texts=[query],
    where={"category": "tech", "date": {"$gte": "2024-01-01"}}
)
```

**4. Sharding**
```python
# Shard by category/tenant
tech_collection = client.get_collection("tech")
science_collection = client.get_collection("science")
```

### Best Practices:

**1. Start Simple**
- Use default HNSW params
- Optimize only when needed
- Measure before optimizing

**2. Monitor Everything**
```python
- Query latency (p50, p95, p99)
- Indexing throughput
- Cache hit rate
- Error rate
- Disk usage
```

**3. Test at Scale**
```python
# Load test with production-like data
- 1M+ vectors
- Realistic query patterns
- Concurrent users
```

**4. Plan for Growth**
```python
# Design for 10x current size
- Horizontal scaling ready
- Storage capacity planning
- Cost projections
```

### Common Pitfalls:

❌ **Don't:**
- Use in-memory DB in production
- Add documents one-by-one
- Ignore tail latency (p99)
- Skip backups
- Over-optimize early

✅ **Do:**
- Use persistent storage
- Batch operations
- Monitor all percentiles
- Regular backups
- Optimize based on metrics

### Next Steps:

You've completed the Embeddings & Vectors module!

**Next module:** Continue with remaining modules